# 01 - Data cleaning

**Client question:** are our promotions actually increasing sales, in which stores and situations, and which should we keep, change or stop?

This notebook turns the raw Rossmann files into one clean table (`data/clean/sales_clean.csv`) and writes a cleaning log (`docs/cleaning_log.md`).
Raw files are never modified. Every step records how many rows it changed and why.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

SEED = 42
RAW, CLEAN, DOCS = Path("../data/raw"), Path("../data/clean"), Path("../docs")
CLEAN.mkdir(parents=True, exist_ok=True)
DOCS.mkdir(parents=True, exist_ok=True)

log = []  # one entry per cleaning step


def add_log(step, rows_before, rows_after, why):
    log.append({"step": step, "rows_before": rows_before, "rows_after": rows_after,
                "rows_changed": rows_after - rows_before, "why": why})

## 1. Load raw data

In [2]:
# StateHoliday is read as text: it mixes 0 and the letters a/b/c, which pandas would otherwise read inconsistently
train = pd.read_csv(RAW / "train.csv", dtype={"StateHoliday": str}, parse_dates=["Date"])
store = pd.read_csv(RAW / "store.csv")

print("train:", train.shape, "| store:", store.shape)
print("\nmissing in train:\n", train.isna().sum()[lambda s: s > 0])
print("\nmissing in store:\n", store.isna().sum()[lambda s: s > 0])

train: (1017209, 9) | store: (1115, 10)

missing in train:
 Series([], dtype: int64)

missing in store:
 CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64


## 2. Sample 50 stores (stratified by store type)

The full data is ~1M rows; a 50-store sample keeps the project fast. A plain random sample would almost never pick store type `b` (only 17 of 1,115 stores),
which would make any "by store type" finding impossible, so I fix how many stores come from each type. Fixed seed, so it is reproducible.

In [3]:
ALLOC = {"a": 24, "b": 4, "c": 8, "d": 14}  # 50 stores; b is deliberately over-represented so it can be analysed
print(store.StoreType.value_counts().to_dict(), "<- stores available per type")

sample_stores = pd.concat(
    [store[store.StoreType == t].sample(n, random_state=SEED) for t, n in ALLOC.items()]
)
assert sample_stores.Store.nunique() == 50
sample_stores.StoreType.value_counts().to_dict()

{'a': 602, 'd': 348, 'c': 148, 'b': 17} <- stores available per type


{'a': 24, 'd': 14, 'c': 8, 'b': 4}

## 3. Merge daily sales with store attributes

In [4]:
n_train = len(train)
df = train[train.Store.isin(sample_stores.Store)].merge(
    sample_stores, on="Store", how="left", validate="m:1"  # many daily rows -> one store row
)
add_log("Sample 50 stores and merge store.csv", n_train, len(df),
        "Keep a small stratified sample; merge so each daily row carries its store's type, assortment and competition info")
print(df.shape, "| stores:", df.Store.nunique(), "| dates:", df.Date.min().date(), "to", df.Date.max().date())

(45444, 18) | stores: 50 | dates: 2013-01-01 to 2015-07-31


## 4. Duplicates

In [5]:
n_dup = df.duplicated(["Store", "Date"]).sum()
before = len(df)
df = df.drop_duplicates(["Store", "Date"])
add_log("Drop duplicate Store+Date rows", before, len(df),
        "One store should have one row per day; duplicates would double-count sales")
print("duplicate Store+Date rows:", n_dup)

duplicate Store+Date rows: 0


## 5. Fix types and inconsistent categories

`StateHoliday` mixes `0` (no holiday) with letters. I turn it into readable labels so it is one consistent text column.
`PromoInterval` is blank for stores without Promo2, so I label it `None` instead of leaving it missing.

In [6]:
print(df.StateHoliday.value_counts().to_dict())
holiday_map = {"0": "none", "a": "public", "b": "easter", "c": "christmas"}
df["StateHoliday"] = df.StateHoliday.map(holiday_map)
assert df.StateHoliday.notna().all(), "unexpected StateHoliday value"

df["PromoInterval"] = df.PromoInterval.fillna("None")
for c in ["Open", "Promo", "SchoolHoliday", "Promo2"]:
    df[c] = df[c].astype(int)

add_log("Standardise StateHoliday and PromoInterval", len(df), len(df),
        "Make categorical columns consistent (0/a/b/c -> none/public/easter/christmas; blank -> None); no rows removed")
df.dtypes

{'0': 44045, 'a': 917, 'b': 300, 'c': 182}


Store                                 int64
DayOfWeek                             int64
Date                         datetime64[us]
Sales                                 int64
Customers                             int64
Open                                  int64
Promo                                 int64
StateHoliday                            str
SchoolHoliday                         int64
StoreType                               str
Assortment                              str
CompetitionDistance                 float64
CompetitionOpenSinceMonth           float64
CompetitionOpenSinceYear            float64
Promo2                                int64
Promo2SinceWeek                     float64
Promo2SinceYear                     float64
PromoInterval                           str
dtype: object

## 6. Closed days and zero-sales days

A closed store has zero sales because it was shut, not because a promo failed. Keeping those rows would drag the average down and could
make promotions look worse or better than they are, so they are removed. Then I check for open days with zero sales, which look like recording errors.

In [7]:
print("closed days:", (df.Open == 0).sum(), "| closed days with sales > 0:", ((df.Open == 0) & (df.Sales > 0)).sum())
before = len(df)
df = df[df.Open == 1]
add_log("Drop closed-store days", before, len(df),
        "Sales are zero because the store was shut, which says nothing about promotion effectiveness")

zero_open = (df.Sales == 0).sum()
before = len(df)
df = df[df.Sales > 0]
add_log("Drop open days with zero sales", before, len(df),
        "An open store with zero sales looks like a recording error, not real demand")
print("open days with zero sales:", zero_open)

closed days: 7405 | closed days with sales > 0: 0
open days with zero sales: 2


## 7. Missing values

`CompetitionDistance` is missing for a handful of stores; I fill it with the median (robust to a few very distant competitors) and add a flag column
so the fill is never hidden. `CompetitionOpenSince*` and `Promo2Since*` are blank when there is no competitor or no Promo2, so they are left as they are.

In [8]:
median_dist = store.CompetitionDistance.median()  # from the full store table so the value is stable, not sample-dependent
n_missing = df.CompetitionDistance.isna().sum()
df["CompetitionDistance_missing"] = df.CompetitionDistance.isna().astype(int)
df["CompetitionDistance"] = df.CompetitionDistance.fillna(median_dist)

add_log("Fill missing CompetitionDistance with median", len(df), len(df),
        f"{n_missing} rows filled with median {median_dist:.0f} m; flag column CompetitionDistance_missing keeps the fill visible")
print("rows filled:", n_missing, "| median used:", median_dist)
print("\nremaining missing values:\n", df.isna().sum()[lambda s: s > 0])

rows filled: 0 | median used: 2325.0

remaining missing values:
 CompetitionOpenSinceMonth    11687
CompetitionOpenSinceYear     11687
Promo2SinceWeek              19031
Promo2SinceYear              19031
dtype: int64


## 8. Outliers: flag, don't delete

A very high sales day can be a genuine event (e.g. a big promo weekend), so deleting it would remove exactly the effect we want to measure.
I flag days far above that store's normal level (more than 3 IQRs above its 75th percentile) and keep them. The model stage can test its sensitivity to them.

In [9]:
def above_upper_fence(s):
    q1, q3 = s.quantile([0.25, 0.75])
    return s > q3 + 3 * (q3 - q1)


df["is_sales_outlier"] = df.groupby("Store").Sales.transform(above_upper_fence).astype(int)
n_out = df.is_sales_outlier.sum()
add_log("Flag Sales outliers (kept)", len(df), len(df),
        f"{n_out} rows flagged as is_sales_outlier=1 (>3 IQR above the store's Q3); kept because they may be real promo effects")
print("flagged:", n_out, f"({n_out / len(df):.2%})")

flagged: 38 (0.10%)


## 9. Date parts, final checks and save

In [10]:
df["Year"] = df.Date.dt.year
df["Month"] = df.Date.dt.month
df = df.sort_values(["Store", "Date"]).reset_index(drop=True)

# final checks: fail loudly if anything is off
assert not df.duplicated(["Store", "Date"]).any()
assert (df.Sales > 0).all() and (df.Open == 1).all()
assert df.CompetitionDistance.notna().all()
assert df.Store.nunique() == 50

df.to_csv(CLEAN / "sales_clean.csv", index=False)
print("saved", df.shape, "->", CLEAN / "sales_clean.csv")
print("promo share of rows:", round(df.Promo.mean(), 3))
df.head()

saved (38037, 22) -> ../data/clean/sales_clean.csv
promo share of rows: 0.442


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,CompetitionDistance_missing,is_sales_outlier,Year,Month
0,5,3,2013-01-02,4253,577,1,0,none,1,a,...,4.0,2015.0,0,NaN,NaN,None,0,0,2013,1
1,5,4,2013-01-03,3465,491,1,0,none,0,a,...,4.0,2015.0,0,NaN,NaN,None,0,0,2013,1
2,5,5,2013-01-04,4456,533,1,0,none,0,a,...,4.0,2015.0,0,NaN,NaN,None,0,0,2013,1
3,5,6,2013-01-05,1590,202,1,0,none,0,a,...,4.0,2015.0,0,NaN,NaN,None,0,0,2013,1
4,5,1,2013-01-07,6978,717,1,1,none,0,a,...,4.0,2015.0,0,NaN,NaN,None,0,0,2013,1


## 10. Write the cleaning log

In [11]:
lines = [
    "# Cleaning log",
    "",
    "Generated by `notebooks/01_cleaning.ipynb`. Raw files in `data/raw/` are untouched.",
    "",
    f"- Source: Rossmann Store Sales (Kaggle), `train.csv` ({n_train:,} rows) and `store.csv` (1,115 stores)",
    "- Sample: 50 stores, stratified by StoreType (a=24, b=4, c=8, d=14), seed 42",
    f"- Output: `data/clean/sales_clean.csv` ({len(df):,} rows, {df.shape[1]} columns)",
    "",
    "| # | Step | Rows before | Rows after | Change | Why |",
    "|---|------|------------:|-----------:|-------:|-----|",
]
for i, e in enumerate(log, 1):
    lines.append(f"| {i} | {e['step']} | {e['rows_before']:,} | {e['rows_after']:,} | {e['rows_changed']:+,} | {e['why']} |")

lines += [
    "",
    "## Decisions to defend",
    "- **Closed days removed, not kept as zeros:** zero sales on a closed day is not a promotion failure, and keeping it would bias promo vs non-promo comparisons.",
    "- **Median fill + flag for CompetitionDistance:** median is robust to extreme distances; the flag column keeps the imputation visible to the model and the reader.",
    "- **Outliers flagged, not deleted:** big sales days may be real promo effects; removing them would understate uplift.",
    "- **Stratified sample:** without fixing per-type counts, store type `b` (17 of 1,115 stores) would almost never appear.",
    "",
    "## Known limitations",
    "- 50 stores is a sample; results describe this sample and should be re-run on the full data before any real decision.",
    "- `CompetitionOpenSince*` and `Promo2Since*` are blank where the competitor/Promo2 does not exist; they are not used yet.",
    "- Data ends 2015-07-31; no information on promotion cost, so any money impact in the final memo is revenue, not profit.",
]
(DOCS / "cleaning_log.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines[:16]))

# Cleaning log

Generated by `notebooks/01_cleaning.ipynb`. Raw files in `data/raw/` are untouched.

- Source: Rossmann Store Sales (Kaggle), `train.csv` (1,017,209 rows) and `store.csv` (1,115 stores)
- Sample: 50 stores, stratified by StoreType (a=24, b=4, c=8, d=14), seed 42
- Output: `data/clean/sales_clean.csv` (38,037 rows, 22 columns)

| # | Step | Rows before | Rows after | Change | Why |
|---|------|------------:|-----------:|-------:|-----|
| 1 | Sample 50 stores and merge store.csv | 1,017,209 | 45,444 | -971,765 | Keep a small stratified sample; merge so each daily row carries its store's type, assortment and competition info |
| 2 | Drop duplicate Store+Date rows | 45,444 | 45,444 | +0 | One store should have one row per day; duplicates would double-count sales |
| 3 | Standardise StateHoliday and PromoInterval | 45,444 | 45,444 | +0 | Make categorical columns consistent (0/a/b/c -> none/public/easter/christmas; blank -> None); no rows removed |
| 4 | Drop closed-store day